# Milestone 1

This milestone helps to familiarize me with the process of Exploratory Data Analysis (EDA), text processing, and establishing baseline similarity metrics required specifically for Natural Language Processing (NLP) pipelines.  

Topic Readings:

- Tokenization & Text Normalization  

- Stop Words & Vocabulary Constraints  

- TF-IDF (Term Frequency-Inverse Document Frequency)  

- Cosine Similarity in NLP 

- Understanding Mean Average Precision (MAP@3)  

---

---

In [13]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [14]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


##### Question 1: 

Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [15]:
answer_count = train['answer'].value_counts().sort_index()
print(answer_count)

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64


In [16]:
sum_result = answer_count.max() + answer_count.min()
print(sum_result)

814


#### Question 2:

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [19]:
import string

cleaned_prompts = train['prompt'].str.lower()
cleaned_prompts = cleaned_prompts.apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))

unique_words = set()
for prompt in cleaned_prompts:
    words = prompt.split()
    unique_words.update(words)

vocabulary_size = len(unique_words)
print(vocabulary_size)

859


#### Question 3:

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  


In [24]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

cleaned_prompt_row1 = cleaned_prompts.iloc[0]
words_row1 = cleaned_prompt_row1.split()

filtered_words = [word for word in words_row1 if word not in ENGLISH_STOP_WORDS]

remaining_words_count = len(filtered_words)
print(remaining_words_count)
print(filtered_words)

13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


#### Question 4:

Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer? 

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_texts = (train['prompt']+' '+train['A']+' '+train['B']+' '+train['C']+' '+train['D']+' '+train['E']).tolist()
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_texts)
tfidf_vocab_size = X.shape[1]
print(tfidf_vocab_size)

2762


#### Question 5:

Using the TF-IDF vectorizer fitted in Question 4, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

prompt_text = train.iloc[0]['prompt']
option_a = train.iloc[0]['A']

v_prompt = vectorizer.transform([prompt_text])
v_option_a = vectorizer.transform([option_a])

sim_score = cosine_similarity(v_prompt, v_option_a)[0, 0]
print(f"{sim_score:.4f}")

0.2720


#### Question 6:

Expand the logic from Question 5: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [29]:
similarities = []

for idx, row in train.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])
    
    sim_a = cosine_similarity(prompt_vec, vectorizer.transform([row['A']]))[0, 0]
    sim_b = cosine_similarity(prompt_vec, vectorizer.transform([row['B']]))[0, 0]
    sim_c = cosine_similarity(prompt_vec, vectorizer.transform([row['C']]))[0, 0]
    sim_d = cosine_similarity(prompt_vec, vectorizer.transform([row['D']]))[0, 0]
    sim_e = cosine_similarity(prompt_vec, vectorizer.transform([row['E']]))[0, 0]
    
    sims = {'A': sim_a, 'B': sim_b, 'C': sim_c, 'D': sim_d, 'E': sim_e}
    predicted_answer = max(sims, key=sims.get)
    
    is_correct = predicted_answer == row['answer']
    similarities.append(is_correct)

correct_count = sum(similarities)
total_count = len(train)
percentage = (correct_count / total_count) * 100

print(f"Correct predictions: {correct_count}/{total_count}")
print(f"Percentage: {percentage:.2f}%")

Correct predictions: 271/2000
Percentage: 13.55%


#### Question 7:

If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [30]:
ground_truth = 'C'
prediction = ['C', 'A', 'B']

if ground_truth in prediction[:3]:
    rank = prediction.index(ground_truth) + 1
    map3 = 1.0 / rank
else:
    map3 = 0.0

print(map3)

1.0


#### Question 8:

If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [32]:
ground_truth = 'B'
prediction = ['D', 'B', 'E']

if ground_truth in prediction[:3]:
    rank = prediction.index(ground_truth) + 1
    map3 = 1.0 / rank
else:
    map3 = 0.0

print(map3)

0.5


#### Question 9:

The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [33]:
top_answers = answer_count.sort_values(ascending=False).index[:3].tolist()

def map3_for_answer(true_answer):
    if true_answer in top_answers:
        rank = top_answers.index(true_answer) + 1
        return 1.0 / rank
    return 0.0

map3_scores = train['answer'].apply(map3_for_answer)
majority_baseline_map3 = map3_scores.mean()

print("Top 3 static prediction order:", top_answers)
print("Majority class baseline MAP@3:", majority_baseline_map3)

Top 3 static prediction order: ['B', 'C', 'A']
Majority class baseline MAP@3: 0.42125


#### Question 10:

The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [34]:
def map3(true_answer, ranked_answers):
    if true_answer in ranked_answers[:3]:
        return 1.0 / (ranked_answers[:3].index(true_answer) + 1)
    return 0.0

map3_scores = []

for _, row in train.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])
    option_scores = [
        ('A', cosine_similarity(prompt_vec, vectorizer.transform([row['A']]))[0, 0]),
        ('B', cosine_similarity(prompt_vec, vectorizer.transform([row['B']]))[0, 0]),
        ('C', cosine_similarity(prompt_vec, vectorizer.transform([row['C']]))[0, 0]),
        ('D', cosine_similarity(prompt_vec, vectorizer.transform([row['D']]))[0, 0]),
        ('E', cosine_similarity(prompt_vec, vectorizer.transform([row['E']]))[0, 0]),
    ]
    ranked_answers = [opt for opt, _ in sorted(option_scores, key=lambda x: x[1], reverse=True)]
    map3_scores.append(map3(row['answer'], ranked_answers))

tfidf_pipeline_map3 = np.mean(map3_scores)
print(tfidf_pipeline_map3)

0.2961666666666667
